# Notebook 2: Phylogenetic Tree -- Neighbor-Joining Algorithm

**Author:** Chidera Ibe (coibe2)

**Project:** Evolutionary Conservation of the PI3K-alpha Cryptic Drug Binding Site

---

In this notebook we construct a phylogenetic tree of PI3K-alpha orthologs using
the **Neighbor-Joining (NJ)** algorithm (Saitou & Nei, 1987). The tree will
serve two purposes:

1. **Guide tree** for progressive multiple sequence alignment (Notebook 3).
2. **Evolutionary framework** for interpreting conservation patterns at the
   cryptic binding site.

### Workflow

1. Load ortholog protein sequences (fetched in Notebook 1, or re-fetched here).
2. Select ~20 representative species spanning vertebrate diversity.
3. Compute an all-vs-all pairwise distance matrix via Needleman-Wunsch
   alignments with Jukes-Cantor correction.
4. Build a tree with the Neighbor-Joining algorithm.
5. Visualize and verify the tree topology.

---
## Section 1: Setup & Data Loading

In [ ]:
# ---- path setup ----
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ---- imports ----
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

from src.fetchers import (
    fetch_ensembl_orthologs,
    fetch_ensembl_sequences,
    fetch_human_pik3ca_sequence,
)
from src.utils import load_blosum62, jukes_cantor_distance, read_fasta, write_fasta
from src.alignment import needleman_wunsch, compute_distance_matrix
from src.tree import neighbor_joining

print("All imports successful.")

### Load ortholog sequences

We attempt to load cached ortholog data produced by Notebook 1. If the cache
is unavailable, we re-fetch from Ensembl.

In [ ]:
# Fetch orthologs from Ensembl (cached after first call)
orthologs = fetch_ensembl_orthologs(gene_symbol="PIK3CA", species="human")
print(f"Total orthologs returned by Ensembl: {len(orthologs)}")

# Build a species -> protein_id mapping (prefer one2one orthologs)
species_protein = {}
for orth in orthologs:
    sp = orth["species"]
    # Prefer one2one over one2many
    if sp not in species_protein or orth["type"] == "ortholog_one2one":
        species_protein[sp] = orth["protein_id"]

print(f"Unique species with orthologs: {len(species_protein)}")

### Select ~20 representative species

We choose species that span the major vertebrate lineages to give the tree
good taxonomic coverage while keeping computation tractable (pairwise NW
alignment is O(L^2) per pair, and we need n*(n-1)/2 pairs).

In [ ]:
# Target species (Ensembl naming convention: genus_species, lowercase)
TARGET_SPECIES = {
    # Primates
    "homo_sapiens":            "Human",
    "pan_troglodytes":         "Chimp",
    "macaca_mulatta":          "Macaque",
    # Rodents
    "mus_musculus":            "Mouse",
    "rattus_norvegicus":       "Rat",
    # Lagomorphs
    "oryctolagus_cuniculus":   "Rabbit",
    # Carnivores
    "canis_lupus_familiaris":  "Dog",
    "felis_catus":             "Cat",
    # Ungulates
    "bos_taurus":              "Cow",
    "sus_scrofa":              "Pig",
    "equus_caballus":          "Horse",
    # Marsupial
    "monodelphis_domestica":   "Opossum",
    # Monotreme
    "ornithorhynchus_anatinus":"Platypus",
    # Birds
    "gallus_gallus":           "Chicken",
    "meleagris_gallopavo":     "Turkey",
    # Reptile
    "anolis_carolinensis":     "Lizard",
    # Amphibian
    "xenopus_tropicalis":      "Frog",
    # Fish
    "danio_rerio":             "Zebrafish",
    "takifugu_rubripes":       "Fugu",
    "latimeria_chalumnae":     "Coelacanth",
}

# Also try alternative Ensembl names for dog
ALT_NAMES = {
    "canis_lupus_familiaris": ["canis_familiaris"],
}

# Match target species against available orthologs
selected_proteins = {}  # common_name -> protein_id
missing = []

for ensembl_name, common_name in TARGET_SPECIES.items():
    if ensembl_name in species_protein:
        selected_proteins[common_name] = species_protein[ensembl_name]
    else:
        # Try alternative names
        found = False
        for alt in ALT_NAMES.get(ensembl_name, []):
            if alt in species_protein:
                selected_proteins[common_name] = species_protein[alt]
                found = True
                break
        if not found:
            missing.append((common_name, ensembl_name))

print(f"Selected {len(selected_proteins)} / {len(TARGET_SPECIES)} target species.")
if missing:
    print(f"Missing species (not in Ensembl orthologs): {missing}")
print()
for name, pid in selected_proteins.items():
    print(f"  {name:12s}  {pid}")

In [ ]:
# Fetch protein sequences for selected species
protein_ids = list(selected_proteins.values())
raw_sequences = fetch_ensembl_sequences(protein_ids)

# Also fetch human sequence directly (in case it's not in ortholog set)
human_seq = fetch_human_pik3ca_sequence()

# Build the final dict: common_name -> sequence
sequences = {}
for common_name, pid in selected_proteins.items():
    if pid in raw_sequences:
        sequences[common_name] = raw_sequences[pid]

# Ensure human is present
if "Human" not in sequences:
    sequences["Human"] = human_seq

print(f"\nLoaded sequences for {len(sequences)} species:\n")
print(f"{'Species':12s}  {'Length':>6s}")
print("-" * 22)
for name, seq in sorted(sequences.items()):
    print(f"{name:12s}  {len(seq):6d}")

---
## Section 2: Pairwise Distance Matrix

We compute all-vs-all pairwise global alignments using the **Needleman-Wunsch**
algorithm with the BLOSUM62 substitution matrix and affine gap penalties
(open = -10, extend = -0.5). From each alignment we calculate the fraction of
differing positions, then apply the **Jukes-Cantor correction** to obtain
evolutionary distances:

$$
d = -\frac{19}{20} \ln\left(1 - \frac{20}{19} p\right)
$$

where $p$ is the proportion of differing aligned residues. This correction
accounts for the possibility of multiple substitutions at the same site and is
appropriate for protein sequences (20 amino acid states).

In [ ]:
# Load substitution matrix
blosum62 = load_blosum62()
print("BLOSUM62 loaded.")

In [ ]:
# Compute the distance matrix
# This uses compute_distance_matrix() from src.alignment, which internally
# runs NW alignment for every pair and applies Jukes-Cantor correction.
#
# With ~20 species of ~1068 aa each, we have ~190 NW alignments.
# Each NW alignment is O(L^2) ~ O(1e6), so total ~ 2e8 operations.
# This may take several minutes.

species_names = sorted(sequences.keys())
ordered_sequences = {name: sequences[name] for name in species_names}

n_species = len(species_names)
n_pairs = n_species * (n_species - 1) // 2
print(f"Computing {n_pairs} pairwise NW alignments for {n_species} species...")
print(f"(Each alignment involves sequences of ~{np.mean([len(s) for s in sequences.values()]):.0f} aa)")
print()

# We use our own loop instead of compute_distance_matrix() to add progress reporting
labels = list(ordered_sequences.keys())
n = len(labels)
dist_matrix = np.zeros((n, n))
pair_count = 0

for i in range(n):
    for j in range(i + 1, n):
        pair_count += 1
        if pair_count % 10 == 0 or pair_count == 1:
            print(f"  Pair {pair_count:3d}/{n_pairs}: {labels[i]} vs {labels[j]}")

        aln1, aln2, score = needleman_wunsch(
            ordered_sequences[labels[i]],
            ordered_sequences[labels[j]],
            blosum62,
            gap_open=-10,
            gap_extend=-0.5,
        )

        # Compute fraction of differing aligned positions
        aligned_pos = sum(1 for a, b in zip(aln1, aln2) if a != "-" or b != "-")
        identical = sum(1 for a, b in zip(aln1, aln2) if a == b and a != "-")
        p = 1.0 - (identical / aligned_pos) if aligned_pos > 0 else 1.0

        d = jukes_cantor_distance(p)
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d

print(f"\nDone! Computed {pair_count} pairwise distances.")

In [ ]:
# Display the distance matrix as a heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    dist_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Jukes-Cantor Distance"},
    ax=ax,
)
ax.set_title("Pairwise Evolutionary Distances (Jukes-Cantor corrected)", fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 3: Neighbor-Joining Algorithm

The **Neighbor-Joining (NJ)** algorithm (Saitou & Nei, 1987) is a
distance-based method for constructing phylogenetic trees. Unlike the simpler
UPGMA algorithm, NJ does **not** assume a molecular clock (equal rates of
evolution in all lineages), making it more appropriate for real data.

### Algorithm Steps

Given an $n \times n$ distance matrix $D$:

1. **Compute net divergence** $r_i = \sum_{k=1}^{n} D_{ik}$ for each taxon $i$.

2. **Compute the Q-matrix:**
   $$Q_{ij} = (n-2) \cdot D_{ij} - r_i - r_j$$
   The Q-matrix adjusts pairwise distances by the average divergence of each
   taxon, so that taxa that are generally distant from everything else are
   penalized less.

3. **Find the pair $(i, j)$ with minimum $Q_{ij}$** -- these are the next
   neighbors to be joined.

4. **Compute branch lengths** from the new internal node $u$ to each of the
   joined taxa:
   $$d(i, u) = \frac{D_{ij}}{2} + \frac{r_i - r_j}{2(n-2)}$$
   $$d(j, u) = D_{ij} - d(i, u)$$

5. **Update the distance matrix** -- replace taxa $i$ and $j$ with the new
   node $u$, computing distances to all remaining taxa:
   $$D_{uk} = \frac{D_{ik} + D_{jk} - D_{ij}}{2}$$

6. **Repeat** until only 2 nodes remain, then connect them.

The result is an unrooted tree in Newick format.

In [ ]:
# Run Neighbor-Joining
print("Running Neighbor-Joining algorithm...")
newick_str = neighbor_joining(dist_matrix, labels)
print("\nNewick string:")
print(newick_str)

In [ ]:
# Save the tree to disk
tree_dir = os.path.join(PROJECT_ROOT, "data", "processed", "trees")
os.makedirs(tree_dir, exist_ok=True)
tree_path = os.path.join(tree_dir, "guide_tree.nwk")

with open(tree_path, "w") as f:
    f.write(newick_str + "\n")

print(f"Guide tree saved to: {tree_path}")

---
## Section 4: Tree Visualization

We visualize the tree in two ways:
1. A quick rectangular phylogram via `Bio.Phylo.draw()`.
2. A publication-quality figure with branches colored by taxonomic group.

In [ ]:
from Bio import Phylo

# Parse the Newick string
tree = Phylo.read(StringIO(newick_str), "newick")

# Basic rectangular phylogram
fig, ax = plt.subplots(figsize=(10, 8))
Phylo.draw(tree, axes=ax, do_show=False)
ax.set_title("PI3K-alpha Ortholog Phylogeny (Neighbor-Joining)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def get_taxonomic_group(species_name):
    """
    Assign a taxonomic group and color to a species based on its common name.

    Returns
    -------
    group : str
    color : str (matplotlib color)
    """
    name = species_name.lower() if species_name else ""

    primates = ["human", "chimp", "macaque"]
    rodents = ["mouse", "rat"]
    birds = ["chicken", "turkey"]
    reptiles = ["lizard"]
    amphibians = ["frog"]
    fish = ["zebrafish", "fugu", "coelacanth"]
    # Everything else is "other mammal"

    if name in primates:
        return "Primates", "#2171b5"        # blue
    elif name in rodents:
        return "Rodents", "#238b45"          # green
    elif name in birds:
        return "Birds", "#cb181d"            # red
    elif name in reptiles:
        return "Reptiles", "#6a51a3"         # purple
    elif name in amphibians:
        return "Amphibians", "#8c6d31"       # brown
    elif name in fish:
        return "Fish", "#08979c"             # teal
    else:
        return "Other mammals", "#e6550d"    # orange


# Collect all terminal names and assign colors
terminal_colors = {}
for clade in tree.get_terminals():
    if clade.name:
        group, color = get_taxonomic_group(clade.name)
        terminal_colors[clade.name] = (group, color)

print("Taxonomic group assignments:")
for name, (group, color) in sorted(terminal_colors.items()):
    print(f"  {name:12s} -> {group}")

In [ ]:
def color_tree_clades(tree, terminal_colors):
    """
    Assign colors to all clades. Terminal clades get their taxonomic color;
    internal clades get the color of their children if all children share
    the same color, otherwise gray.
    """
    def get_clade_color(clade):
        if clade.is_terminal():
            if clade.name and clade.name in terminal_colors:
                return terminal_colors[clade.name][1]
            return "#555555"

        child_colors = set()
        for child in clade.clades:
            child_colors.add(get_clade_color(child))

        if len(child_colors) == 1:
            return child_colors.pop()
        return "#555555"  # gray for mixed groups

    # Assign color attribute to each clade
    for clade in tree.find_clades():
        clade.color = get_clade_color(clade)


color_tree_clades(tree, terminal_colors)

# Draw with colors
fig, ax = plt.subplots(figsize=(12, 9))

# Use Bio.Phylo.draw with label_colors
label_colors = {}
for clade in tree.get_terminals():
    if clade.name and clade.name in terminal_colors:
        label_colors[clade.name] = terminal_colors[clade.name][1]

Phylo.draw(
    tree,
    axes=ax,
    do_show=False,
    label_colors=label_colors,
)

ax.set_title(
    "PI3K-alpha Ortholog Phylogeny (Neighbor-Joining)\n"
    "Branches colored by taxonomic group",
    fontsize=13,
)

# Add legend
from matplotlib.lines import Line2D
group_colors = {
    "Primates":       "#2171b5",
    "Rodents":        "#238b45",
    "Other mammals":  "#e6550d",
    "Birds":          "#cb181d",
    "Reptiles":       "#6a51a3",
    "Amphibians":     "#8c6d31",
    "Fish":           "#08979c",
}
legend_elements = [
    Line2D([0], [0], color=c, lw=2, label=g)
    for g, c in group_colors.items()
]
ax.legend(
    handles=legend_elements,
    loc="lower left",
    fontsize=9,
    title="Taxonomic Group",
    framealpha=0.9,
)

plt.tight_layout()

# Save figure
results_dir = os.path.join(PROJECT_ROOT, "data", "results")
os.makedirs(results_dir, exist_ok=True)
fig_path = os.path.join(results_dir, "phylogenetic_tree.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")

plt.show()

---
## Section 5: Verification

We verify the tree topology against well-established vertebrate phylogeny.
Key expectations:

- **Human and Chimp** should be nearest neighbors (diverged ~6 Mya).
- **Mouse and Rat** should be nearest neighbors (diverged ~12 Mya).
- **Chicken and Turkey** should cluster together (both Galliformes).
- Fish should be outgroups to all tetrapods.
- Mammals should form a clade separate from birds/reptiles.

These checks help us confirm that our from-scratch NW + NJ pipeline is
producing biologically meaningful results.

In [ ]:
def find_nearest_neighbor(tree, taxon_name):
    """
    Find the nearest neighbor of a taxon in a Phylo tree.
    The nearest neighbor is the other leaf in the smallest clade containing
    the target taxon and exactly one other leaf.
    """
    # Get the path from root to the target
    target = None
    for leaf in tree.get_terminals():
        if leaf.name == taxon_name:
            target = leaf
            break
    if target is None:
        return None

    # Find the smallest clade containing the target with exactly 2 terminals
    for clade in tree.find_clades(order="level"):
        terminals = clade.get_terminals()
        names = [t.name for t in terminals]
        if taxon_name in names and len(names) == 2:
            other = [n for n in names if n != taxon_name][0]
            return other
    return None


def check_cluster(tree, taxon1, taxon2):
    """
    Check whether two taxa are nearest neighbors (i.e., share a parent
    clade that contains only these two terminals).
    """
    for clade in tree.find_clades(order="level"):
        terminals = [t.name for t in clade.get_terminals()]
        if len(terminals) == 2 and taxon1 in terminals and taxon2 in terminals:
            return True
    return False


def check_monophyly(tree, taxa_list):
    """
    Check whether a set of taxa forms a monophyletic group (clade).
    Returns True if there exists a clade whose terminal set equals taxa_list.
    """
    taxa_set = set(taxa_list)
    for clade in tree.find_clades(order="level"):
        clade_taxa = set(t.name for t in clade.get_terminals())
        if clade_taxa == taxa_set:
            return True
    return False

In [ ]:
# Re-parse tree to ensure clean state
tree_check = Phylo.read(StringIO(newick_str), "newick")

# Get available terminal names for checking
available_taxa = set(t.name for t in tree_check.get_terminals())
print("Terminal taxa in tree:", sorted(available_taxa))
print()

# ----- Topology checks -----
checks = []

# Check 1: Human-Chimp nearest neighbors
if "Human" in available_taxa and "Chimp" in available_taxa:
    result = check_cluster(tree_check, "Human", "Chimp")
    checks.append(("Human-Chimp nearest neighbors", result))
    nn_human = find_nearest_neighbor(tree_check, "Human")
    print(f"Human nearest neighbor: {nn_human}")

# Check 2: Mouse-Rat nearest neighbors
if "Mouse" in available_taxa and "Rat" in available_taxa:
    result = check_cluster(tree_check, "Mouse", "Rat")
    checks.append(("Mouse-Rat nearest neighbors", result))
    nn_mouse = find_nearest_neighbor(tree_check, "Mouse")
    print(f"Mouse nearest neighbor: {nn_mouse}")

# Check 3: Chicken-Turkey cluster
if "Chicken" in available_taxa and "Turkey" in available_taxa:
    result = check_cluster(tree_check, "Chicken", "Turkey")
    checks.append(("Chicken-Turkey cluster", result))
    nn_chicken = find_nearest_neighbor(tree_check, "Chicken")
    print(f"Chicken nearest neighbor: {nn_chicken}")

# Check 4: Primates monophyly
primate_taxa = [t for t in ["Human", "Chimp", "Macaque"] if t in available_taxa]
if len(primate_taxa) >= 2:
    result = check_monophyly(tree_check, primate_taxa)
    checks.append((f"Primates monophyly ({', '.join(primate_taxa)})", result))

# Check 5: Rodents monophyly
rodent_taxa = [t for t in ["Mouse", "Rat"] if t in available_taxa]
if len(rodent_taxa) == 2:
    result = check_monophyly(tree_check, rodent_taxa)
    checks.append((f"Rodents monophyly ({', '.join(rodent_taxa)})", result))

print()
print("=" * 55)
print("  TOPOLOGY VERIFICATION SUMMARY")
print("=" * 55)
for desc, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}]  {desc}")

n_pass = sum(1 for _, p in checks if p)
n_total = len(checks)
print(f"\n  Result: {n_pass}/{n_total} checks passed.")

if n_pass == n_total:
    print("  The NJ tree topology is consistent with known vertebrate phylogeny.")
else:
    print("  Some checks failed. This may indicate long-branch attraction or")
    print("  limited phylogenetic signal at the protein level for closely related taxa.")
    print("  The tree is still suitable as a guide tree for MSA.")

In [ ]:
# Optional: Robinson-Foulds distance comparison (requires ete3)
try:
    from ete3 import Tree as EteTree

    # Our NJ tree
    nj_ete = EteTree(newick_str)

    # Expected vertebrate topology (simplified, unrooted)
    # This is a rough expected topology based on established vertebrate phylogeny
    expected_taxa = sorted(available_taxa)
    print("ete3 available -- computing Robinson-Foulds distance.")

    # Build a reference tree for the taxa we have
    # (A full reference would require all our taxa; we compare against
    #  the standard mammalian/vertebrate topology)
    # For now, just report basic tree statistics
    print(f"  NJ tree leaves: {len(nj_ete)}")
    print(f"  NJ tree internal nodes: {len(list(nj_ete.traverse())) - len(nj_ete)}")

    # If a reference tree is available, compute RF distance:
    # rf, max_rf, _, _, _, _, _ = nj_ete.robinson_foulds(ref_ete)
    # print(f"  Robinson-Foulds distance: {rf} / {max_rf}")
    print("  (Full RF comparison requires a reference tree with matching taxa.)")

except ImportError:
    print("ete3 not installed -- skipping Robinson-Foulds distance calculation.")
    print("Install with: pip install ete3")

---
## Summary

In this notebook we:

1. **Loaded** PI3K-alpha ortholog sequences for ~20 representative vertebrate species.
2. **Computed** an all-vs-all pairwise distance matrix using Needleman-Wunsch
   global alignment (BLOSUM62, affine gaps) with Jukes-Cantor distance correction.
3. **Built** a phylogenetic tree using the Neighbor-Joining algorithm
   (implemented from scratch in `src/tree.py`).
4. **Visualized** the tree with taxonomic-group coloring.
5. **Verified** the tree topology against known vertebrate phylogeny.

The resulting guide tree is saved to `data/processed/trees/guide_tree.nwk` and
will be used in **Notebook 3** for progressive multiple sequence alignment.

### Next Steps

- **Notebook 3:** Use this guide tree to perform progressive MSA of all orthologs.
- **Notebook 4:** Map conservation scores onto the PI3K-alpha 3D structure,
  focusing on the cryptic binding site.